# HF TDOA Analysis - Figure 12

This notebook uses the `hf_tdoa_lib` library to analyze HF TDOA measurements for AB5YO on 40m and 60m bands.

In [1]:
import os
import hf_tdoa_lib as tdoa

%matplotlib inline

# Setup plotting style
tdoa.setup_plotting_style()

## Load WAV Files & Find Chirps for AB5YO 40m

In [ ]:
base_dir   = 'data'
data_set_40m = 'TX_WA5FRF_EL09nn-RX_AB5YO_EL09so-40m'
sweep_rate = 10  # Hz/ms

# Path to chirp template - used for finding chirp locations via cross-correlation
template_40m = os.path.join('templates', 'PRN_40m_template.wav')

data_dir_40m = os.path.join(base_dir, data_set_40m)
wavlist_40m  = tdoa.obtain_wav_list(data_dir_40m)

In [ ]:
# Correlate each WAV with a known template chirp to identify chirp locations in each WAV file.
chirps_40m = tdoa.find_chirps(wavlist_40m, template_40m, sweep_rate=sweep_rate, plot_correlation=False)

## Find TDOAs for AB5YO 40m

In [ ]:
debug_TDOAs = False

# Process the 2F2-1F2 mode for 40m
chirps_40m = tdoa.find_TDOAs(chirps_40m, mode_string='2F2-1F2',
                             plot_fft=debug_TDOAs, only_one=debug_TDOAs)

# Build the TDOA configuration dictionary
tdoa_dct_40m = tdoa.build_tdoa_config(chirps_40m, mode_strings=['2F2-1F2'])

## Load WAV Files & Find Chirps for AB5YO 60m

In [ ]:
data_set_60m = 'TX_WA5FRF_EL09nn-RX_AB5YO_EL09so-60m'

# Path to chirp template for 60m
template_60m = os.path.join('templates', 'PRN_60m_template.wav')

data_dir_60m = os.path.join(base_dir, data_set_60m)
wavlist_60m  = tdoa.obtain_wav_list(data_dir_60m)

In [ ]:
# Correlate each WAV with the 60m template chirp
chirps_60m = tdoa.find_chirps(wavlist_60m, template_60m, sweep_rate=sweep_rate, plot_correlation=False)

## Find TDOAs for AB5YO 60m

In [ ]:
# Process the 2F2-1F2 mode for 60m
chirps_60m = tdoa.find_TDOAs(chirps_60m, mode_string='2F2-1F2',
                             plot_fft=debug_TDOAs, only_one=debug_TDOAs)

# Build the TDOA configuration dictionary
tdoa_dct_60m = tdoa.build_tdoa_config(chirps_60m, mode_strings=['2F2-1F2'])

## Create Figure 12: AB5YO 40m and 60m Subplots

Figure 12 shows layer height measurements from AB5YO on both 40m (top panel) and 60m (bottom panel) bands.

In [ ]:
# Get path_info for solar calculations - use AB5YO location
path_info_40m = chirps_40m.attrs['path_info']
solar_lat, solar_lon = path_info_40m.get_midpoint()

# Model coefficients for AB5YO (from OLD notebook: slope=150, intercept=0)
model_coeffs_AB5YO = (150, 0)

# Create the subplot figure with both 40m and 60m
tdoa.plot_hmf2_subplot(
    chirps_list=[chirps_40m, chirps_60m],
    tdoa_dct_list=[tdoa_dct_40m, tdoa_dct_60m],
    subplot_labels=['(a)', '(b)'],
    ylim=(200, 350),
    solar_lat=solar_lat,
    solar_lon=solar_lon,
    overlay_solar_elevation=True,
    overlay_eclipse=True,
    ionosonde_dct={'overlay_hmE': False},
    tdoa_csv_dct_list=[
        {
            'csv_path': 'data/CSVs/2024-04-08_TX_WA5FRF_EL09nn-RX_AB5YO_EL09so-40m_TDOA.csv',
            'model_coeffs': model_coeffs_AB5YO
        },
        {
            'csv_path': 'data/CSVs/2024-04-08_TX_WA5FRF_EL09nn-RX_AB5YO_EL09so-60m_TDOA.csv',
            'model_coeffs': model_coeffs_AB5YO
        }
    ],
    figsize=(15, 16)
)